## MVP ANA — Semantic Search over STJ Súmulas

### Project Overview

This notebook implements a 10-step NLP pipeline that transforms the STJ Súmulas PDF into a semantic search index.

**Pipeline Summary:**
1. Read raw text from the STJ Súmulas PDF (with OCR fallback)
2. Normalize encoding artifacts (mojibake, smart quotes)
3. Remove repeated boilerplate headers using TF-IDF similarity
4. Segment the document into one block per Súmula
5. Normalize legal citations to typed placeholder tokens
6. Label each Súmula segment with its legal area, sub-area, and Súmula number
7. Generate BERTimbau embeddings (mean-pooled, sliding window)
8. Build a FAISS semantic search index and execute queries
9. Visualize cosine similarity distributions per query
10. Estimate query occurrence probability via KDE and softmax ranking

**Domain:** Brazilian Superior Court of Justice (STJ) — Súmulas are binding legal summaries grouped by legal area.

**Reference:** [STJ Súmulas PDF](https://www.stj.jus.br/docs_internet/VerbetesSTJ_asc.pdf)

### Import Libraries

Install required packages and import all pipeline modules. Modules are reloaded to pick up source changes without restarting the kernel.

| Package group | Purpose |
|---------------|---------|
| `pdfplumber`, `pytesseract`, `Pillow` | PDF text extraction and OCR fallback for scanned pages |
| `ftfy`, `scikit-learn` | Encoding repair and TF-IDF boilerplate detection |
| `transformers`, `torch` | BERTimbau model loading and inference |
| `numpy`, `faiss-cpu` | Numerical computation and vector similarity search |
| `matplotlib` | Figure generation for similarity and probability visualizations |
| `scipy` | Gaussian KDE and softmax for probabilistic query ranking |

In [1]:
%pip install pdfplumber pytesseract Pillow ftfy scikit-learn transformers torch numpy matplotlib faiss-cpu scipy

import warnings
import logging
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

scripts_path = Path("./pipeline/scripts")
if str(scripts_path.resolve()) not in sys.path:
    sys.path.insert(0, str(scripts_path.resolve()))

import pipeline_step, pipeline_manager
import step_0_pdf_reader, step_1_encoding_normalizer, step_2_boilerplate_remover
import step_3_sentence_segmenter, step_4_citation_normalizer
import step_5_labeler, step_6_embedding_generator
import step_7_search_index, step_8_similarity_visualizer
import step_9_probability_estimator

for mod in [pipeline_step, pipeline_manager,
            step_0_pdf_reader, step_1_encoding_normalizer, step_2_boilerplate_remover,
            step_3_sentence_segmenter, step_4_citation_normalizer,
            step_5_labeler, step_6_embedding_generator,
            step_7_search_index, step_8_similarity_visualizer,
            step_9_probability_estimator]:
    importlib.reload(mod)

from pipeline_step import PipelineStep
from pipeline_manager import PipelineManager
from step_0_pdf_reader import PdfReader, PdfReaderInput
from step_1_encoding_normalizer import EncodingNormalizer
from step_2_boilerplate_remover import BoilerplateRemover
from step_3_sentence_segmenter import SentenceSegmenter
from step_4_citation_normalizer import CitationNormalizer
from step_5_labeler import SumulaLabeler
from step_6_embedding_generator import EmbeddingGenerator
from step_7_search_index import SearchIndexBuilder
from step_8_similarity_visualizer import SimilarityVisualizer
from step_9_probability_estimator import ProbabilityEstimator

Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'matplotlib.colormaps'

### Pipeline Configuration

All tunable parameters are declared here as a single source of truth. Change values in this cell before re-running the pipeline — downstream cells reference these variables directly.

In [ ]:
random_seed = 42  # PARAMETER: single source of randomness

pdf_path = Path("./datasets/Súmulas - STJ.pdf")  # PARAMETER: input PDF
ocr_fallback = True  # PARAMETER: OCR fallback for scanned pages
min_page_text = 50  # PARAMETER: min chars per page before OCR

tfidf_threshold = 0.92  # PARAMETER: boilerplate detection threshold
min_sentence_tokens = 5  # PARAMETER: min tokens per segment

embedding_batch_size = 16  # PARAMETER: sentences per inference batch
max_tokens = 512  # PARAMETER: BERT context window
overlap_tokens = 64  # PARAMETER: sliding window overlap
embedding_model_name = "neuralmind/bert-large-portuguese-cased"  # PARAMETER: embedding model

search_queries = [  # PARAMETER: queries to compare
    "banco cobrou taxa de juros abusiva no contrato",
    "acidente de trânsito causou dano material ao veículo",
    "servidor público foi demitido por ato administrativo ilegal",
    "consumidor pediu rescisão do contrato de plano de saúde",
    "herdeiro contestou partilha de bens no inventário",
]
search_top_k = 500  # PARAMETER: number of results per query (covers full corpus)

checkpoint_dir = Path("./pipeline/checkpoints")
figures_dir = Path("./pipeline/figures")

### Build Pipeline

Instantiate all pipeline steps in execution order and register them with `PipelineManager`. The manager uses pickle-based checkpoints so that expensive steps (especially step 6 — BERTimbau inference) are loaded from disk on subsequent runs instead of re-executing.

> **WARNING — Destructive operation:** The `clear_checkpoints(from_step=...)` call below **permanently deletes** checkpoint files for all steps at or after the specified step number. Checkpoints for expensive steps such as step 6 (BERTimbau inference) will need to be recomputed from scratch. Change `from_step` only when you intentionally want to invalidate cached outputs.

In [ ]:
steps = [
    PdfReader(min_text_length=min_page_text),
    EncodingNormalizer(),
    BoilerplateRemover(tfidf_threshold=tfidf_threshold),
    SentenceSegmenter(min_tokens=min_sentence_tokens),
    CitationNormalizer(),
    SumulaLabeler(),
    EmbeddingGenerator(
        max_tokens=max_tokens,
        overlap_tokens=overlap_tokens,
        batch_size=embedding_batch_size,
        model_name=embedding_model_name,
    ),
    SearchIndexBuilder(queries=search_queries, top_k=search_top_k),
    SimilarityVisualizer(output_dir=figures_dir),
    ProbabilityEstimator(output_dir=figures_dir),
]
manager = PipelineManager(checkpoint_dir=checkpoint_dir, steps=steps)
manager.clear_checkpoints(from_step=7)  # PARAMETER: first step to invalidate — increase to preserve more cached outputs
status = manager.checkpoint_status()
print("Checkpoint status:")
for step_num, exists in status.items():
    print(f"  Step {step_num:2d}: {'cached' if exists else 'pending'}")

### Step 0. Read Text from PDF

Extracts raw text from the STJ Súmulas PDF using `pdfplumber`. Pages with fewer than `min_page_text` characters are re-processed with Tesseract OCR to recover scanned content. The raw text is stored as a single string for downstream normalization.

In [ ]:
pdf_input = PdfReaderInput(pdf_path=pdf_path, ocr_fallback=ocr_fallback, min_text_length=min_page_text)
step0_output = manager.run_step(0, pdf_input)
print(f"Pages processed: {step0_output.page_count}")
print(f"OCR pages: {len(step0_output.ocr_pages)}")
print(f"Total characters: {len(step0_output.raw_text):,}")

### Step 1. Encoding Normalization

Repairs mojibake and encoding artifacts introduced by the PDF text layer (e.g., curly quotes, em-dashes, Latin-1 sequences misread as UTF-8). `ftfy` is used to fix common encoding issues deterministically.

In [ ]:
step1_output = manager.run_step(1, step0_output)
print(f"Replacements applied: {len(step1_output.replacements)}")

### Step 2. Boilerplate Removal

Detects and replaces repeated page headers and footers with a `BOILERPLATE_TOKEN` sentinel. TF-IDF cosine similarity between adjacent text blocks identifies repetitions above `tfidf_threshold`. This preserves the delimiter structure needed for segmentation in the next step.

In [ ]:
step2_output = manager.run_step(2, step1_output)
print(f"Segments replaced: {step2_output.removed_count}")

### Step 3. Súmula Segmentation

Splits the filtered text into individual Súmula blocks using `BOILERPLATE_TOKEN` boundaries as natural delimiters. Each page header position in the original PDF separates consecutive Súmulas, so the token positions become reliable split points. Falls back to double-newline paragraph splitting when fewer than two boilerplate boundaries are found.

In [ ]:
step3_output = manager.run_step(3, step2_output)
print(f"Súmula segments extracted: {step3_output.sentence_count}")
print(f"\nSample segment (first 300 chars):\n{step3_output.sentences[0][:300]!r}")

### Step 4. Citation Normalization

Replaces legal citation strings (article references, process numbers, case references, Súmula references) with typed placeholder tokens such as `<ART_REF>` and `<CASE_REF>`. This prevents downstream embeddings from treating idiosyncratic citation strings as meaningful semantic features while preserving the original text in `citation_metadata` for traceability.

In [ ]:
step4_output = manager.run_step(4, step3_output)
total_citations = sum(len(m) for m in step4_output.citation_metadata)
print(f"Total citations normalized: {total_citations}")

### Step 5. Súmula Labeling

Each Súmula segment is labeled using the document header lines (area + sub-area) that appear before the `Súmula N` declaration line. The STJ PDF groups Súmulas under bold legal area headings (e.g., `BANCÁRIO`) and sub-area headings (e.g., `Contrato Bancário`). Labels follow the pattern `Area_Sub_area_N` with accent-stripped title-case sanitization.

In [ ]:
step5_output = manager.run_step(5, step4_output)
print(f"Labeled súmulas: {len(step5_output.labeled_sentences)}")
print(f"\nSample labels (first 10):")
for ls in step5_output.labeled_sentences[:10]:
    print(f"  {ls.label}")

### Step 6. Embedding Generator

[STEP CAN BE SKIPPED - re-runs use checkpoint]

Encodes each labeled Súmula segment into a dense vector using the BERTimbau model (`neuralmind/bert-large-portuguese-cased`). Mean pooling over all non-padding token hidden states produces a single vector per segment. Segments exceeding 512 tokens are handled via a sliding window with `overlap_tokens` overlap; window embeddings are averaged. Label metadata is propagated into `EmbeddedSentence` for downstream use.

In [ ]:
step6_output = manager.run_step(6, step5_output)
print(f"Model: {step6_output.model_name}")
print(f"Embedding dim: {step6_output.embedding_dim}")
print(f"Embedded sentences: {len(step6_output.embedded_sentences)}")

### Step 7. Semantic Search Index

FAISS inner-product index over L2-normalized BERTimbau embeddings. At query time the Portuguese input is encoded with the same model and the top-K most similar Súmulas are returned with their label and similarity score. Cosine similarity is equivalent to inner product on L2-normalized vectors, so `faiss.IndexFlatIP` is the correct choice. The step persists its output as a checkpoint so the FAISS build and BERTimbau query encoding are skipped on re-runs.

In [ ]:
step7_output = manager.run_step(7, step6_output)

print(f"Queries run: {len(step7_output)}")
for o in step7_output:
    print(f"\n  Query: {o.query!r}")
    print(f"  n={len(o.results)}  mean={o.mean_similarity:.4f}  median={o.median_similarity:.4f}  max={o.max_similarity:.4f}  min={o.min_similarity:.4f}")
    print(f"  std={o.std_similarity:.4f}  var={o.variance_similarity:.5f}  range={o.range_similarity:.4f}  IQR={o.iqr_similarity:.4f}  CV={o.cv_similarity:.4f}")
    print(f"  P5={o.percentile(5):.4f}  P25={o.percentile(25):.4f}  P75={o.percentile(75):.4f}  P90={o.percentile(90):.4f}  P95={o.percentile(95):.4f}")


### Step 8. Similarity Intensity Visualization

A single figure with five vertically stacked histograms is produced — one per query, sorted descending by mean similarity.

Each subplot shows the full frequency distribution of cosine similarities across all retrieved Súmulas. All subplots share the same X axis (Cosine Similarity). Bars are stacked and colored by legal area label.

#### Statistics displayed per histogram

| Statistic | Symbol | Description |
|-----------|--------|-------------|
| **n** | count | Number of súmulas retrieved for this query |
| **mean** | μ | Arithmetic mean similarity — higher means the query is broadly well-matched across the corpus |
| **median** | m | Middle value of the sorted similarities — robust center unaffected by outliers |
| **max** | max | Cosine similarity of the single best-matching súmula (top-1 score) |
| **min** | min | Cosine similarity of the worst match in the result set |
| **std** | σ | Standard deviation — how spread out the similarities are around the mean |
| **var** | σ² | Variance — square of std; raw dispersion measure |
| **range** | max−min | Full spread of the distribution |
| **IQR** | Q3−Q1 | Interquartile range — spread of the middle 50% of results, robust to outliers |
| **CV** | σ/μ | Coefficient of variation — relative dispersion; low CV means consistent similarity across results |
| **P5/P25/P75/P90/P95** | percentiles | Values at the 5th, 25th, 75th, 90th, and 95th percentiles — reveal the shape and skew of the distribution without summary compression |

#### Interpreting query relevance

- **High mean + low CV** → query is consistently well-matched across many súmulas (broadly represented theme)
- **High max + high std** → query has a few excellent matches but most results are mediocre (niche but precise theme)
- **Low mean + low range** → query is uniformly weakly matched (theme not well represented in the corpus)

The figure is saved to `figures_dir` as `similarity_histogram.png`.

In [ ]:
step8_output = manager.run_step(8, step7_output)

for fig in step8_output.figures:
    plt.tight_layout()
    plt.show()
print(f"Figures saved: {[str(p) for p in step8_output.saved_paths]}")

### Step 9. Query Occurrence Probability Estimation

Answers the central question: **"Which query is most likely to appear as a case theme in the STJ Súmulas document?"**

Three complementary probabilistic metrics are combined into a single ranked score:

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **Softmax Probability** | `exp(mean_sim) / Σ exp(mean_sim_j)` | Relative probability of each query given mean similarity to corpus |
| **KDE Probability** | `∫[corpus_mean → max] KDE(x) dx` | Probability mass above the corpus baseline under a Gaussian kernel density estimate — threshold-free |
| **Z-Score** | `(mean_sim − corpus_mean) / σ` | How many standard deviations above the corpus-wide baseline the query's mean similarity sits |

**Combined score** = `0.4 × softmax + 0.4 × KDE + 0.2 × Z-score` (each normalized to [0, 1]).

The winner is highlighted in green on the ranking chart.

In [ ]:
step9_output = manager.run_step(9, step7_output)

print("=" * 60)
print("QUERY OCCURRENCE PROBABILITY RANKING")
print("=" * 60)
for p in step9_output.query_probabilities:
    marker = " ← WINNER" if p.rank == 1 else ""
    print(f"\n  Rank {p.rank}{marker}")
    print(f"  Query:    {p.query!r}")
    print(f"  Softmax:  {p.softmax_probability:.4f}")
    print(f"  KDE:      {p.kde_probability:.4f}")
    print(f"  Z-Score:  {p.z_score:.4f}")
    print(f"  Combined: {p.combined_score:.4f}")
print(f"\ncorpus_mean={step9_output.corpus_mean:.4f}  corpus_std={step9_output.corpus_std:.4f}")

for fig in step9_output.figures:
    plt.tight_layout()
    plt.show()
print(f"Figures saved: {[str(p) for p in step9_output.saved_paths]}")

### Considerations

- **Label extraction** depends on consistent document header formatting in the STJ PDF. Non-standard sections may fall back to `Desconhecido_Desconhecido_N`.
- **Semantic similarity** is cosine distance in BERTimbau embedding space. Súmulas that are lexically distant but thematically close will rank highly, which is the intended behavior for cross-area legal reasoning.
- **Query language** must be Portuguese. English or mixed-language queries will not align with the embedding space trained on Portuguese legal text.
- **Checkpoint reuse** means that changing `embedding_model_name` or earlier step parameters requires clearing the affected checkpoint files before re-running.
- **FAISS index** is rebuilt in memory on each session from the step 6 checkpoint; it is not persisted separately.

### Conclusions

The 10-step pipeline transforms the STJ Súmulas PDF into an interactive semantic search tool without any clustering dependency:

| Step | Component | Role |
|------|-----------|------|
| 0 | PdfReader | Extract raw text with OCR fallback |
| 1 | EncodingNormalizer | Repair encoding artifacts |
| 2 | BoilerplateRemover | Mark repeated headers as delimiters |
| 3 | SentenceSegmenter | Split into per-Súmula blocks |
| 4 | CitationNormalizer | Replace citations with typed tokens |
| 5 | SumulaLabeler | Annotate area + sub-area + number |
| 6 | EmbeddingGenerator | BERTimbau mean-pooled vectors |
| 7 | SearchIndexBuilder | FAISS cosine similarity search with checkpoint |
| 8 | SimilarityVisualizer | Cosine similarity histogram |
| 9 | ProbabilityEstimator | KDE + Softmax + Z-score query occurrence ranking |

The label-aware search index enables targeted retrieval of Súmulas by legal area while the visualizer reveals where semantic intensity concentrates across the complete 502-Súmula corpus for any Portuguese legal query.